# Лабораторная работа по Spark: инкрементальная загрузка хранилища данных

Этот notebook является самостоятельным артефактом и не использует модули из `src`.

В тетради содержится только минимальный код, необходимый для трёх задач:

1. Построение физической модели данных на основе информации об источниках данных.
2. Построение parquet-хранилища данных.
3. Разработка процессов инкрементального обновления распределённых данных при использовании Apache Spark.


In [1]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import re
import shutil
import sys

from pathlib import Path
from typing import Any


# Единый реестр источников: здесь зафиксирована физическая модель входных данных.
SOURCES = {
    'client_cards_daily': {
        'source_id': 1,
        'source_description': 'Ежедневный источник по клиентским картам.',
        'update_frequency': 'daily',
        'target_table': 'client_daily_attrs_scd2',
        'partition_key': 'row_actual_to',
        # columns описывает атрибуты, которые будут разворачиваться в EAV-структуру.
        'columns': (
            'srv_mb_nflag',
            'cc_stoplist',
            'lne_tot_debt_int_ovrd_rub_amt',
            'lne_tot_debt_ovrd_rub_amt',
        ),
        # attribute_data_types нужен при заполнении справочника dim_attributes.
        'attribute_data_types': {
            'srv_mb_nflag': 'INT',
            'cc_stoplist': 'INT',
            'lne_tot_debt_int_ovrd_rub_amt': 'DECIMAL',
            'lne_tot_debt_ovrd_rub_amt': 'DECIMAL',
        },
    },
    'credit_cards_info': {
        'source_id': 2,
        'source_description': 'Ежемесячный источник по кредитным картам.',
        'update_frequency': 'monthly',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'columns': (
            'client_income_amt',
            'oi_total_amt',
            'act_pl_os_rub_amt',
            'payroll_client_nflag',
            'inf_payroll_rub_amt',
            'legal_entity_amt',
            'inc_avg_risk_rub_amt',
            'otf_loan_rub_amt',
            'otf_fee_rub_amt',
            'inf_transfer_rub_amt',
            'cc_ever_nflag',
        ),
        'attribute_data_types': {
            'client_income_amt': 'DECIMAL',
            'oi_total_amt': 'DECIMAL',
            'act_pl_os_rub_amt': 'DECIMAL',
            'payroll_client_nflag': 'INT',
            'inf_payroll_rub_amt': 'DECIMAL',
            'legal_entity_amt': 'DECIMAL',
            'inc_avg_risk_rub_amt': 'DECIMAL',
            'otf_loan_rub_amt': 'DECIMAL',
            'otf_fee_rub_amt': 'DECIMAL',
            'inf_transfer_rub_amt': 'DECIMAL',
            'cc_ever_nflag': 'INT',
        },
    },
    'deb_cards_info': {
        'source_id': 3,
        'source_description': 'Ежемесячный источник по дебетовым картам.',
        'update_frequency': 'monthly',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'columns': (
            'onl_bank_active_1m_nfalg',
            'auto_pay_active_qty',
            'cl_income_1m_amt',
            'dep_acc_1st_open_dt',
            'wdr_cash_6m_amt',
            'cash_op_6m_amt',
            'cash_3m_qty',
            'lst_balance_amt',
            'card_active_1m_nflag',
        ),
        'attribute_data_types': {
            'onl_bank_active_1m_nfalg': 'INT',
            'auto_pay_active_qty': 'INT',
            'cl_income_1m_amt': 'DECIMAL',
            'dep_acc_1st_open_dt': 'DATE',
            'wdr_cash_6m_amt': 'DECIMAL',
            'cash_op_6m_amt': 'DECIMAL',
            'cash_3m_qty': 'INT',
            'lst_balance_amt': 'DECIMAL',
            'card_active_1m_nflag': 'INT',
        },
    },
}


In [2]:
def configure_java_for_spark() -> None:
    """
    Настраивает переменные окружения Java для локального запуска Spark на macOS.

    Функция подставляет Homebrew-путь к OpenJDK, если он есть в системе, и добавляет
    Java-опции, необходимые для совместимой работы Spark/Hadoop на современных JDK.
    """
    # Используем стандартный путь Homebrew к Java, который ожидается на локальной машине.
    java_home = Path('/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home')
    if java_home.exists():
        os.environ['JAVA_HOME'] = str(java_home)
        java_bin = str(java_home / 'bin')
        path_parts = os.environ.get('PATH', '').split(os.pathsep)
        # Не дублируем путь к java bin при повторных запусках ячейки.
        if java_bin not in path_parts:
            os.environ['PATH'] = os.pathsep.join([java_bin, *path_parts])

    # Эти опции нужны, чтобы Spark/Hadoop корректно стартовал на современной Java.
    java_options = '-Djava.security.manager=allow'
    for variable_name in ('JAVA_TOOL_OPTIONS', '_JAVA_OPTIONS', 'SPARK_SUBMIT_OPTS'):
        existing_options = os.environ.get(variable_name, '')
        if java_options not in existing_options:
            os.environ[variable_name] = f'{existing_options} {java_options}'.strip()


def ensure_pyspark_available() -> None:
    """
    Проверяет, что текущая Python-среда может импортировать `pyspark`.

    Notebook не устанавливает зависимости самостоятельно. Вместо этого он использует
    стандартный Homebrew `SPARK_HOME`, если тот доступен, и затем делает обычный импорт PySpark.
    """
    configure_java_for_spark()
    spark_home = Path(os.environ.get('SPARK_HOME', '/opt/homebrew/opt/apache-spark/libexec'))
    lib_dir = spark_home / 'python' / 'lib'
    pyspark_zip = lib_dir / 'pyspark.zip'
    py4j_zip = next(lib_dir.glob('py4j-*.zip'), None)
    if pyspark_zip.exists() and py4j_zip is not None:
        os.environ['SPARK_HOME'] = str(spark_home)
        # Подключаем Python-часть системного Spark в текущее окружение notebook.
        for path in (spark_home / 'python', pyspark_zip, py4j_zip):
            path_text = str(path)
            if path_text in sys.path:
                sys.path.remove(path_text)
            sys.path.insert(0, path_text)

    # Явно фиксируем, что и driver, и executor используют текущий интерпретатор Python.
    os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
    os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)
    import pyspark  # noqa: F401


def create_spark_session(
    app_name: str = 'spark-lab',
    warehouse_dir: str | Path | None = None,
    shuffle_partitions: int = 8,
):
    """
    Создаёт локальную `SparkSession` для выполнения лабораторной работы.

    Сессия использует `local[2]`, UTC и file-based parquet-таблицы. При передаче
    `warehouse_dir` Spark получает отдельный metastore-каталог, что делает прогоны воспроизводимыми.
    """
    ensure_pyspark_available()
    from pyspark.sql import SparkSession

    # Отдельно задаём технического пользователя локального Spark/Hadoop-контекста.
    os.environ.setdefault('SPARK_USER', os.environ.get('USER', 'spark'))
    os.environ.setdefault('HADOOP_USER_NAME', os.environ['SPARK_USER'])

    # Собираем минимально необходимую конфигурацию Spark для лабораторной работы.
    builder = (
        SparkSession.builder.master('local[2]')
        .appName(app_name)
        .config('spark.sql.shuffle.partitions', str(shuffle_partitions))
        .config('spark.sql.session.timeZone', 'UTC')
        .config('spark.driver.extraJavaOptions', '-Djava.security.manager=allow')
        .config('spark.executor.extraJavaOptions', '-Djava.security.manager=allow')
        .config('spark.hadoop.fs.file.impl', 'org.apache.hadoop.fs.LocalFileSystem')
        .config('spark.hadoop.fs.viewfs.impl', 'org.apache.hadoop.fs.LocalFileSystem')
        .config('spark.hadoop.hadoop.security.authentication', 'simple')
        .config('spark.hadoop.hadoop.security.authorization', 'false')
    )
    if warehouse_dir is not None:
        # Выносим metastore в отдельную директорию, чтобы разные прогоны не конфликтовали друг с другом.
        builder = builder.config('spark.sql.warehouse.dir', str(Path(warehouse_dir).resolve()))

    spark = builder.getOrCreate()
    user_group_information = spark.sparkContext._jvm.org.apache.hadoop.security.UserGroupInformation
    user_group_information.setLoginUser(user_group_information.createRemoteUser(os.environ['SPARK_USER']))
    # Уменьшаем объём служебных логов в выводе notebook.
    spark.sparkContext.setLogLevel('WARN')
    return spark


## 2. Поиск и контроль партиций источников


In [6]:
def parse_partition_directory_name(partition_dir_name: str) -> tuple[str, str]:
    """
    Разбирает имя директории партиции вида `key='value'` на имя ключа и значение.

    Функция строго валидирует формат, потому что ошибка в имени каталога должна выявляться до того,
    как начнётся чтение крупных parquet-файлов. Это особенно важно при очень больших объёмах данных,
    где позднее обнаружение ошибки приводит к дорогим холостым запускам.

    Параметры:
        partition_dir_name:
            Имя директории партиции, например `report_dt='2024-01-31'`.

    Возвращаемое значение:
        Кортеж `(partition_key, partition_value)`.

    Исключения:
        ValueError: если имя каталога не соответствует требуемому шаблону.

    Пример входа:
        parse_partition_directory_name("report_dt='2024-01-31'")

    Пример выхода:
        ('report_dt', '2024-01-31')
    """
    # Используем явный regex, чтобы в ETL попадали только корректно названные партиционные каталоги.
    match = re.match(r"^(?P<key>[A-Za-z0-9_]+)='(?P<value>.*)'$", partition_dir_name)
    if match is None:
        raise ValueError(
            f"Имя каталога партиции '{partition_dir_name}' не соответствует шаблону key='value'."
        )
    return match.group('key'), match.group('value')


In [8]:
def discover_source_partitions(
    sources_root: str | Path,
    source_registry: dict[str, dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    """
    Находит все доступные партиции источников и возвращает формализованное описание каждой партиции.

    Функция не читает содержимое parquet-файлов. Она опирается только на структуру директорий и тем самым
    остаётся дешёвой даже при очень большом числе партиций. Возвращаемый результат уже содержит все поля,
    необходимые для последующего принятия решения «новая / пропустить / ошибка».

    Параметры:
        sources_root:
            Путь к директории `sources`.
        source_registry:
            Необязательный реестр источников. По умолчанию используется глобальный `SOURCES`.

    Возвращаемое значение:
        Отсортированный список словарей. Каждый элемент содержит `source_name`, `source_id`, `partition_key`,
        `partition_value`, `partition_path`, `target_table` и `update_frequency`.

    Исключения:
        FileNotFoundError: если директория конкретного источника отсутствует.
        ValueError: если имя партиции не соответствует ожидаемому ключу партиции.

    Пример входа:
        discover_source_partitions('/tmp/sources')

    Пример выхода:
        [
            {
                'source_name': 'credit_cards_info',
                'source_id': 2,
                'partition_key': 'report_dt',
                'partition_value': '2024-01-31',
                'partition_path': '/tmp/sources/credit_cards_info/report_dt='2024-01-31'',
                'target_table': 'client_monthly_attrs_scd1',
                'update_frequency': 'monthly'
            }
        ]
    """
    registry = source_registry or SOURCES
    discovered_partitions: list[dict[str, Any]] = []
    sources_root_path = Path(sources_root)

    # Идём по всем источникам из физической модели и проверяем, что у каждого есть каталог на диске.
    for source_name, source_meta in registry.items():
        source_dir = sources_root_path / source_name
        if not source_dir.exists():
            raise FileNotFoundError(f'Отсутствует директория источника: {source_dir}')

        # Каждый дочерний каталог source_dir трактуем как отдельную физическую партицию источника.
        for partition_dir in sorted(path for path in source_dir.iterdir() if path.is_dir() and not path.name.startswith('.')):
            partition_key, partition_value = parse_partition_directory_name(partition_dir.name)
            if partition_key != source_meta['partition_key']:
                raise ValueError(
                    f"Для источника {source_name} ожидался ключ {source_meta['partition_key']}, но найден {partition_key}."
                )
            discovered_partitions.append(
                {
                    'source_name': source_name,
                    'source_id': source_meta['source_id'],
                    'partition_key': partition_key,
                    'partition_value': partition_value,
                    'partition_path': str(partition_dir.resolve()),
                    'target_table': source_meta['target_table'],
                    'update_frequency': source_meta['update_frequency'],
                }
            )

    # Сортируем результат, чтобы порядок обработки партиций был воспроизводимым между запусками.
    return sorted(
        discovered_partitions,
        key=lambda row: (row['source_id'], row['partition_value'], row['partition_path']),
    )


## 3. Справочники и преобразования Spark


В `dim_attributes.data_type` используются явно заданные значения из `SOURCES`, без автоматического определения типа по имени атрибута.

Это делает справочник более прозрачным: тип каждого атрибута является частью описания источника, а не результатом догадки.


In [11]:
def build_spark_schema(table_name: str):
    """
    Собирает схему Spark для таблицы хранилища.
    """
    from pyspark.sql import types as T

    # Все схемы собраны в одном месте, чтобы одинаково использовать их при чтении и записи.
    schemas = {
        'dim_sources': T.StructType([
            T.StructField('source_id', T.IntegerType(), True),
            T.StructField('source_name', T.StringType(), True),
            T.StructField('source_description', T.StringType(), True),
            T.StructField('update_frequency', T.StringType(), True),
            T.StructField('row_create_dtime', T.TimestampType(), True),
            T.StructField('valid_to', T.TimestampType(), True),
            T.StructField('valid_from', T.TimestampType(), True),
            T.StructField('row_update_dtime', T.TimestampType(), True),
        ]),
        'dim_attributes': T.StructType([
            T.StructField('attribute_id', T.IntegerType(), True),
            T.StructField('attribute_name', T.StringType(), True),
            T.StructField('attribute_description', T.StringType(), True),
            T.StructField('data_type', T.StringType(), True),
            T.StructField('source_id', T.IntegerType(), True),
            T.StructField('update_frequency', T.StringType(), True),
            T.StructField('row_create_dtime', T.TimestampType(), True),
            T.StructField('row_update_dtime', T.TimestampType(), True),
        ]),
        'tech_source_partitions': T.StructType([
            T.StructField('source_id', T.IntegerType(), True),
            T.StructField('source_name', T.StringType(), True),
            T.StructField('target_table', T.StringType(), True),
            T.StructField('partition_key', T.StringType(), True),
            T.StructField('partition_value', T.StringType(), True),
            T.StructField('partition_path', T.StringType(), True),
            T.StructField('manifest_fingerprint', T.StringType(), True),
            T.StructField('last_processed_status', T.StringType(), True),
            T.StructField('first_load_id', T.LongType(), True),
            T.StructField('last_load_id', T.LongType(), True),
            T.StructField('row_create_dtime', T.TimestampType(), True),
            T.StructField('row_update_dtime', T.TimestampType(), True),
        ]),
        'load_log': T.StructType([
            T.StructField('load_id', T.LongType(), True),
            T.StructField('source_id', T.IntegerType(), True),
            T.StructField('source_report_dt', T.StringType(), True),
            T.StructField('load_start_dtime', T.TimestampType(), True),
            T.StructField('load_end_dtime', T.TimestampType(), True),
            T.StructField('target_table', T.StringType(), True),
            T.StructField('load_status', T.StringType(), True),
            T.StructField('row_loading_id', T.LongType(), True),
            T.StructField('error_message', T.StringType(), True),
        ]),
        'client_monthly_attrs_scd1': T.StructType([
            T.StructField('client_id', T.IntegerType(), True),
            T.StructField('attribute_id', T.IntegerType(), True),
            T.StructField('report_dt', T.StringType(), True),
            T.StructField('attribute_value', T.StringType(), True),
            T.StructField('source_id', T.IntegerType(), True),
            T.StructField('row_update_dtime', T.TimestampType(), True),
            T.StructField('row_loading_id', T.LongType(), True),
            T.StructField('row_hash_val', T.StringType(), True),
        ]),
        'client_daily_attrs_scd2': T.StructType([
            T.StructField('client_id', T.IntegerType(), True),
            T.StructField('attribute_id', T.IntegerType(), True),
            T.StructField('attribute_value', T.StringType(), True),
            T.StructField('row_actual_from', T.StringType(), True),
            T.StructField('row_actual_to', T.StringType(), True),
            T.StructField('source_id', T.IntegerType(), True),
            T.StructField('row_update_dtime', T.TimestampType(), True),
            T.StructField('row_loading_id', T.LongType(), True),
            T.StructField('row_hash_val', T.StringType(), True),
        ]),
    }
    # Возвращаем схему строго по имени таблицы.
    return schemas[table_name]


def table_has_parquet_files(table_location: str | Path) -> bool:
    """
    Проверяет, есть ли в каталоге таблицы parquet-файлы.
    """
    # Если parquet-файлов нет, таблица физически считается пустой.
    return any(Path(table_location).glob('**/*.parquet'))


def read_warehouse_table(spark, table_name: str, table_locations: dict[str, str]):
    """
    Читает таблицу хранилища из parquet-каталога.
    """
    # Берём путь к таблице из заранее рассчитанного набора table_locations.
    table_location = table_locations[table_name]
    # При наличии parquet-файлов читаем таблицу с фиксированной схемой.
    if table_has_parquet_files(table_location):
        return spark.read.schema(build_spark_schema(table_name)).parquet(table_location)
    # Иначе возвращаем пустой DataFrame нужной структуры.
    return spark.createDataFrame([], schema=build_spark_schema(table_name))


def initialize_warehouse_tables(spark, warehouse_root: str | Path) -> dict[str, str]:
    """
    Подготавливает parquet-каталоги хранилища по физической схеме.
    """
    # Это полный список parquet-таблиц, составляющих физическую модель хранилища.
    table_names = (
        'dim_sources',
        'dim_attributes',
        'load_log',
        'tech_source_partitions',
        'client_monthly_attrs_scd1',
        'client_daily_attrs_scd2',
    )
    partitioned_by = {
        'client_monthly_attrs_scd1': ['report_dt'],
        'client_daily_attrs_scd2': ['row_actual_to'],
    }

    warehouse_root_path = Path(warehouse_root).resolve()
    warehouse_root_path.mkdir(parents=True, exist_ok=True)
    created_locations: dict[str, str] = {}

    # Создаём каталоги таблиц и, если нужно, инициализируем их пустыми parquet-данными.
    for table_name in table_names:
        table_location = warehouse_root_path / table_name
        table_location.mkdir(parents=True, exist_ok=True)
        # Пустые таблицы создаём заранее, чтобы далее с ними можно было работать единообразно.
        if not table_has_parquet_files(table_location):
            writer = spark.createDataFrame([], schema=build_spark_schema(table_name)).write.mode('overwrite').format('parquet')
            if table_name in partitioned_by:
                writer = writer.partitionBy(*partitioned_by[table_name])
            writer.save(str(table_location))
        created_locations[table_name] = str(table_location)

    return created_locations


In [13]:
def upsert_reference_dimensions(spark, load_timestamp: dt.datetime, table_locations: dict[str, str]) -> dict[str, int]:
    """
    Пересобирает маленькие справочники `dim_sources` и `dim_attributes` на основании текущего списка источников.
    """
    # Генерируем surrogate key для каждого атрибута на основе порядка в SOURCES.
    attribute_id_by_name = {
        attribute_name: attribute_id
        for attribute_id, attribute_name in enumerate(
            [
                column_name
                for source_meta in SOURCES.values()
                for column_name in source_meta['columns']
            ],
            start=1,
        )
    }

    # Сначала собираем строки справочника источников.
    dim_sources_rows = []
    for source_name, source_meta in SOURCES.items():
        dim_sources_rows.append(
            {
                'source_id': source_meta['source_id'],
                'source_name': source_name,
                'source_description': source_meta['source_description'],
                'update_frequency': source_meta['update_frequency'],
                'row_create_dtime': load_timestamp,
                'row_update_dtime': load_timestamp,
                'valid_from': load_timestamp,
                'valid_to': dt.datetime(9999, 12, 31),
            }
        )

    # Затем собираем строки справочника атрибутов с типами и source_id.
    dim_attributes_rows = []
    for source_name, source_meta in SOURCES.items():
        for attribute_name in source_meta['columns']:
            dim_attributes_rows.append(
                {
                    'attribute_id': attribute_id_by_name[attribute_name],
                    'attribute_name': attribute_name,
                    'attribute_description': f'Атрибут {attribute_name} из источника {source_name}.',
                    'data_type': source_meta['attribute_data_types'][attribute_name],
                    'source_id': source_meta['source_id'],
                    'update_frequency': source_meta['update_frequency'],
                    'row_create_dtime': load_timestamp,
                    'row_update_dtime': load_timestamp,
                }
            )

    # Оба справочника маленькие, поэтому здесь уместно полное overwrite-обновление.
    spark.createDataFrame(dim_sources_rows, schema=build_spark_schema('dim_sources')).write.mode('overwrite').parquet(
        table_locations['dim_sources']
    )
    spark.createDataFrame(dim_attributes_rows, schema=build_spark_schema('dim_attributes')).write.mode('overwrite').parquet(
        table_locations['dim_attributes']
    )

    return {
        'dim_sources_rows': len(dim_sources_rows),
        'dim_attributes_rows': len(dim_attributes_rows),
    }


In [15]:
def build_manifest_fingerprint(partition_path: str | Path) -> str:
    """
    Строит стабильный fingerprint parquet-манифеста партиции источника.
    """
    # Fingerprint строим по файловому манифесту партиции, а не по содержимому строк.
    partition_dir = Path(partition_path)
    parquet_files = sorted(partition_dir.glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'В директории {partition_dir} не найдено parquet-файлов.')

    # Для каждого parquet-файла фиксируем имя, размер и mtime_ns.
    manifest_rows = []
    for parquet_file in parquet_files:
        stat = parquet_file.stat()
        manifest_rows.append(f'{parquet_file.name}:{stat.st_size}:{stat.st_mtime_ns}')
    return hashlib.sha256('\n'.join(manifest_rows).encode('utf-8')).hexdigest()


def load_partition_state_map(spark, table_locations: dict[str, str]) -> dict[tuple[int, str, str], dict[str, Any]]:
    """
    Загружает `tech_source_partitions` в словарь для решений `new / skip / fail`.
    """
    # Техническая таблица маленькая, поэтому её можно целиком забрать в память.
    state_rows = read_warehouse_table(spark, 'tech_source_partitions', table_locations).collect()
    state_map: dict[tuple[int, str, str], dict[str, Any]] = {}
    for row in state_rows:
        row_dict = row.asDict()
        state_map[(int(row_dict['source_id']), row_dict['partition_value'], row_dict['target_table'])] = row_dict
    return state_map


def determine_partition_action(existing_state: dict[str, Any] | None, current_fingerprint: str) -> str:
    """
    Возвращает `new`, `skip` или `fail` для найденной source partition.
    """
    # Если состояния ещё нет, партиция считается новой.
    if existing_state is None:
        return 'new'
    # Если fingerprint совпадает, партицию можно безопасно пропустить.
    if existing_state['manifest_fingerprint'] == current_fingerprint:
        return 'skip'
    return 'fail'


def merge_partition_state_entry(spark, state_row: dict[str, Any], table_locations: dict[str, str]) -> None:
    """
    Выполняет upsert строки в маленькую parquet-таблицу `tech_source_partitions`.
    """
    from pyspark.sql import Window
    from pyspark.sql import functions as F

    # Техническое состояние обновляем через временный каталог и последующую замену основной директории.
    locations = table_locations
    # Ключ строки технического состояния определяется источником, значением партиции и целевой таблицей.
    key_columns = ['source_id', 'partition_value', 'target_table']
    existing_df = read_warehouse_table(spark, 'tech_source_partitions', locations)
    new_df = spark.createDataFrame([state_row], schema=build_spark_schema('tech_source_partitions'))
    merged_df = (
        existing_df.unionByName(new_df)
        .withColumn(
            '_state_rank',
            F.row_number().over(
                Window.partitionBy(*key_columns).orderBy(
                    F.col('row_update_dtime').desc_nulls_last(),
                    F.col('last_load_id').desc_nulls_last(),
                )
            ),
        )
        .filter(F.col('_state_rank') == 1)
        .drop('_state_rank')
    )
    temp_location = f"{locations['tech_source_partitions']}_tmp_state"
    merged_df.write.mode('overwrite').parquet(temp_location)
    shutil.rmtree(locations['tech_source_partitions'], ignore_errors=True)
    Path(temp_location).rename(locations['tech_source_partitions'])


In [17]:
def build_stage_dataframe(
    spark,
    raw_df,
    source_name: str,
    partition_value: str | None = None,
    table_locations: dict[str, str] | None = None,
    load_id_by_partition_value: dict[str, int] | None = None,
):
    """
    Преобразует широкую таблицу источника в структуру «сущность-атрибут-значение» средствами Spark.
    """
    from pyspark.sql import functions as F

    # Без table_locations функция не сможет прочитать справочник атрибутов и построить target schema.
    if table_locations is None:
        raise ValueError('Передайте table_locations из initialize_warehouse_tables().')

    # Забираем описание источника из физической модели.
    source_meta = SOURCES[source_name]
    source_id = source_meta['source_id']
    partition_column = source_meta['partition_key']
    # Значение партиции надёжнее извлекать из пути файла: наши каталоги содержат кавычки в имени.
    if partition_value is None:
        partition_expr = F.regexp_extract(F.input_file_name(), rf"{partition_column}='([^/]+)'", 1)
    else:
        partition_expr = F.lit(partition_value)

    # Нормализуем client_id: число оставляем, строку переводим в стабильный int-хэш.
    client_id_expr = F.when(
        F.col('id').cast('string').rlike(r'^[+-]?\\d+$'),
        F.col('id').cast('int'),
    ).otherwise(F.pmod(F.crc32(F.col('id').cast('string')), F.lit(2147483647)).cast('int'))

    # Широкую запись разворачиваем в массив пар attribute_name / attribute_value.
    attribute_items = F.array(
        *[
            F.struct(
                F.lit(column_name).alias('attribute_name'),
                F.col(column_name).cast('string').alias('attribute_value'),
            )
            for column_name in source_meta['columns']
        ]
    )
    # Это базовый набор столбцов, из которого формируется stage DataFrame.
    base_columns = [
        client_id_expr.alias('client_id'),
        F.explode(attribute_items).alias('attribute_item'),
        F.col('row_update_dtime').cast('timestamp').alias('row_update_dtime'),
        F.col('row_hash_val').cast('string').alias('row_hash_val'),
        partition_expr.cast('string').alias('partition_value'),
    ]
    if source_meta['update_frequency'] == 'daily':
        base_columns.insert(2, F.col('row_actual_from').cast('string').alias('row_actual_from'))

    staged_rows = raw_df.select(*base_columns).select(
        'client_id',
        F.col('attribute_item.attribute_name').alias('attribute_name'),
        F.col('attribute_item.attribute_value').alias('attribute_value'),
        *(['row_actual_from'] if source_meta['update_frequency'] == 'daily' else []),
        'row_update_dtime',
        'row_hash_val',
        'partition_value',
    )

    # При пакетной загрузке явно проставляем row_loading_id для каждой партиции.
    if load_id_by_partition_value:
        load_rows = [
            {'partition_value': str(partition_key), 'row_loading_id': int(load_id)}
            for partition_key, load_id in load_id_by_partition_value.items()
        ]
        load_ids_df = spark.createDataFrame(load_rows)
        staged_rows = staged_rows.join(load_ids_df, on='partition_value', how='inner')
    else:
        staged_rows = staged_rows.withColumn('row_loading_id', F.lit(None).cast('bigint'))

    # Соединяем stage-строки со справочником, чтобы получить attribute_id.
    attrs = read_warehouse_table(spark, 'dim_attributes', table_locations).select('source_id', 'attribute_id', 'attribute_name')
    joined = staged_rows.join(
        attrs.filter(F.col('source_id') == F.lit(source_id)),
        on='attribute_name',
        how='inner',
    )

    # Monthly и daily источники пишутся в разные целевые структуры.
    if source_meta['update_frequency'] == 'monthly':
        return joined.select(
            'client_id',
            'attribute_id',
            F.col('partition_value').alias('report_dt'),
            'attribute_value',
            F.lit(source_id).alias('source_id'),
            'row_update_dtime',
            'row_loading_id',
            'row_hash_val',
        )

    return joined.select(
        'client_id',
        'attribute_id',
        'attribute_value',
        'row_actual_from',
        F.col('partition_value').alias('row_actual_to'),
        F.lit(source_id).alias('source_id'),
        'row_update_dtime',
        'row_loading_id',
        'row_hash_val',
    )


## 4. Запись фактов и технических журналов


In [19]:
def append_partitions_to_fact_table(
    spark,
    partition_records: list[dict[str, Any]],
    load_id_by_partition_value: dict[str, int],
    table_locations: dict[str, str],
) -> dict[tuple[str, str, str], int]:
    """
    Читает пачку новых партиций одного источника и записывает их в целевую факт-таблицу в режиме добавления.
    """
    from pyspark.sql import functions as F

    # Пустой список партиций обрабатывать не требуется.
    if not partition_records:
        return {}

    # Batch-loader рассчитан на партиции только одного источника и одной целевой таблицы.
    source_names = {record['source_name'] for record in partition_records}
    target_tables = {record['target_table'] for record in partition_records}
    partition_keys = {record['partition_key'] for record in partition_records}
    if len(source_names) != 1 or len(target_tables) != 1 or len(partition_keys) != 1:
        raise ValueError('Пакетная загрузка допускает только партиции одного источника и одной целевой таблицы.')

    source_name = partition_records[0]['source_name']
    target_table = partition_records[0]['target_table']
    partition_key = partition_records[0]['partition_key']
    partition_paths = [record['partition_path'] for record in partition_records]
    source_root = str(Path(partition_paths[0]).resolve().parent)
    target_partition_column = 'report_dt' if SOURCES[source_name]['update_frequency'] == 'monthly' else 'row_actual_to'

    # Читаем выбранные partition-каталоги через basePath, чтобы Spark сам восстановил partition column.
    raw_df = spark.read.option('basePath', source_root).parquet(*partition_paths)
    stage_df = build_stage_dataframe(
        spark,
        raw_df,
        source_name,
        partition_value=None,
        table_locations=table_locations,
        load_id_by_partition_value=load_id_by_partition_value,
    )
    # Считаем, сколько строк после нормализации попало в каждую target partition.
    rows_by_partition_value = {
        str(row['partition_value']): int(row['rows_loaded'])
        for row in stage_df.groupBy(F.col(target_partition_column).cast('string').alias('partition_value')).count()
        .withColumnRenamed('count', 'rows_loaded')
        .collect()
    }
    partitioned_by = {
        'client_monthly_attrs_scd1': ['report_dt'],
        'client_daily_attrs_scd2': ['row_actual_to'],
    }.get(target_table, [])
    # Записываем данные append-режимом, потому что загружаем только новые физические партиции.
    writer = stage_df.write.mode('append').format('parquet')
    if partitioned_by:
        writer = writer.partitionBy(*partitioned_by)
    writer.save(table_locations[target_table])

    return {
        (record['source_name'], partition_key, record['partition_value']): rows_by_partition_value.get(record['partition_value'], 0)
        for record in partition_records
    }


def append_partition_to_fact_table(
    spark,
    partition_record: dict[str, Any],
    load_id: int,
    table_locations: dict[str, str],
) -> int:
    """
    Читает одну новую партицию источника и записывает её в целевую факт-таблицу.
    """
    # Для одиночной партиции просто переиспользуем batch-логику.
    rows_by_partition = append_partitions_to_fact_table(
        spark,
        [partition_record],
        load_id_by_partition_value={partition_record['partition_value']: load_id},
        table_locations=table_locations,
    )
    partition_key = (
        partition_record['source_name'],
        partition_record['partition_key'],
        partition_record['partition_value'],
    )
    return rows_by_partition[partition_key]


In [21]:
def append_load_log_entry(spark, load_log_row: dict[str, Any], table_locations: dict[str, str]) -> None:
    """
    Добавляет одну запись в `load_log` через запись parquet в режиме добавления.
    """
    # Каждое событие загрузки немедленно дописываем в технический журнал.
    spark.createDataFrame([load_log_row], schema=build_spark_schema('load_log')).write.mode('append').parquet(
        table_locations['load_log']
    )


In [23]:
def run_warehouse_update(
    spark,
    sources_root: str | Path | None = None,
    warehouse_root: str | Path | None = None,
) -> dict[str, Any]:
    """
    Выполняет полный инкрементальный цикл обновления хранилища на файловых путях данных.

    Функция объединяет три основных этапа лабораторной работы: разрешение физических путей,
    построение parquet-структуры хранилища и обработку новых source partitions средствами Spark.
    """
    # Если пользователь не передал пути явно, используем стандартные каталоги проекта.
    resolved_sources_root = Path(sources_root or Path('data') / 'sources').resolve()
    if not resolved_sources_root.exists():
        raise FileNotFoundError('Не удалось определить каталог с исходными данными sources.')

    # parquet-хранилище по умолчанию создаётся в каталоге warehouse_spark_sql.
    warehouse_path = Path(warehouse_root or 'warehouse_spark_sql').resolve()
    # Гарантируем, что все таблицы физически существуют до начала ETL.
    table_locations = initialize_warehouse_tables(spark, warehouse_path)
    run_timestamp = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
    upsert_reference_dimensions(spark, run_timestamp, table_locations)

    # Загружаем техническое состояние и список доступных source partitions.
    state_map = load_partition_state_map(spark, table_locations)
    discovered_partitions = discover_source_partitions(resolved_sources_root)
    max_existing_load_id = read_warehouse_table(spark, 'load_log', table_locations).groupBy().max('load_id').collect()[0][0]
    current_load_id = int(max_existing_load_id or 0)

    processed_partitions = []
    new_partition_contexts = []
    skipped_partitions = 0
    failed_partitions = 0

    # На этом цикле принимается решение по каждой физической партиции источника.
    for partition_record in discovered_partitions:
        current_load_id += 1
        load_start = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
        loaded_key = (
            partition_record['source_id'],
            partition_record['partition_value'],
            partition_record['target_table'],
        )
        current_fingerprint = build_manifest_fingerprint(partition_record['partition_path'])
        existing_state = state_map.get(loaded_key)
        action = determine_partition_action(existing_state, current_fingerprint)

        # Партиция уже была успешно загружена и не изменилась.
        if action == 'skip':
            skipped_partitions += 1
            load_end = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
            append_load_log_entry(
                spark,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_report_dt': partition_record['partition_value'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': load_end,
                    'target_table': partition_record['target_table'],
                    'load_status': 'skipped',
                    'row_loading_id': current_load_id,
                    'error_message': None,
                },
                table_locations,
            )
            updated_state = dict(existing_state)
            updated_state['last_processed_status'] = 'skipped'
            updated_state['last_load_id'] = current_load_id
            updated_state['row_update_dtime'] = load_end
            merge_partition_state_entry(spark, updated_state, table_locations)
            state_map[loaded_key] = updated_state
            processed_partitions.append(
                {
                    'source_name': partition_record['source_name'],
                    'partition_value': partition_record['partition_value'],
                    'action': 'skipped',
                    'rows_loaded': 0,
                }
            )
            continue

        # Партиция была загружена раньше, но её файловый манифест изменился.
        if action == 'fail':
            failed_partitions += 1
            load_end = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
            append_load_log_entry(
                spark,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_report_dt': partition_record['partition_value'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': load_end,
                    'target_table': partition_record['target_table'],
                    'load_status': 'failed',
                    'row_loading_id': current_load_id,
                    'error_message': 'Source partition manifest changed after successful load.',
                },
                table_locations,
            )
            updated_state = dict(existing_state)
            updated_state['last_processed_status'] = 'failed'
            updated_state['last_load_id'] = current_load_id
            updated_state['row_update_dtime'] = load_end
            merge_partition_state_entry(spark, updated_state, table_locations)
            state_map[loaded_key] = updated_state
            processed_partitions.append(
                {
                    'source_name': partition_record['source_name'],
                    'partition_value': partition_record['partition_value'],
                    'action': 'failed',
                    'rows_loaded': 0,
                }
            )
            continue

        # Новые партиции откладываем в batch-контекст для последующей Spark-загрузки.
        new_partition_contexts.append(
            {
                'load_id': current_load_id,
                'load_start': load_start,
                'loaded_key': loaded_key,
                'manifest_fingerprint': current_fingerprint,
                'partition_key': (
                    partition_record['source_name'],
                    partition_record['partition_key'],
                    partition_record['partition_value'],
                ),
                'partition_record': partition_record,
            }
        )

    # Группируем новые партиции по источнику, чтобы читать и писать их пакетно.
    contexts_by_source: dict[str, list[dict[str, Any]]] = {}
    for context in new_partition_contexts:
        source_name = context['partition_record']['source_name']
        contexts_by_source.setdefault(source_name, []).append(context)

    loaded_partitions = 0
    for source_contexts in contexts_by_source.values():
        batch_records = [context['partition_record'] for context in source_contexts]
        try:
            # Здесь выполняется фактическое чтение новых parquet-партиций и запись в витрины.
            rows_by_partition = append_partitions_to_fact_table(
                spark,
                batch_records,
                load_id_by_partition_value={
                    context['partition_record']['partition_value']: context['load_id']
                    for context in source_contexts
                },
                table_locations=table_locations,
            )
        except Exception as exc:
            failed_partitions += len(source_contexts)
            for context in source_contexts:
                partition_record = context['partition_record']
                append_load_log_entry(
                    spark,
                    {
                        'load_id': context['load_id'],
                        'source_id': partition_record['source_id'],
                        'source_report_dt': partition_record['partition_value'],
                        'load_start_dtime': context['load_start'],
                        'load_end_dtime': dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0),
                        'target_table': partition_record['target_table'],
                        'load_status': 'failed',
                        'row_loading_id': context['load_id'],
                        'error_message': str(exc),
                    },
                    table_locations,
                )
            raise

        for context in source_contexts:
            partition_record = context['partition_record']
            partition_key = context['partition_key']
            rows_loaded = rows_by_partition.get(partition_key, 0)
            loaded_partitions += 1
            load_end = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
            append_load_log_entry(
                spark,
                {
                    'load_id': context['load_id'],
                    'source_id': partition_record['source_id'],
                    'source_report_dt': partition_record['partition_value'],
                    'load_start_dtime': context['load_start'],
                    'load_end_dtime': load_end,
                    'target_table': partition_record['target_table'],
                    'load_status': 'loaded',
                    'row_loading_id': context['load_id'],
                    'error_message': None,
                },
                table_locations,
            )
            # После успешной загрузки обновляем техническое состояние партиции.
            new_state = {
                'source_id': partition_record['source_id'],
                'source_name': partition_record['source_name'],
                'target_table': partition_record['target_table'],
                'partition_key': partition_record['partition_key'],
                'partition_value': partition_record['partition_value'],
                'partition_path': partition_record['partition_path'],
                'manifest_fingerprint': context['manifest_fingerprint'],
                'last_processed_status': 'loaded',
                'first_load_id': context['load_id'],
                'last_load_id': context['load_id'],
                'row_create_dtime': load_end,
                'row_update_dtime': load_end,
            }
            merge_partition_state_entry(spark, new_state, table_locations)
            state_map[context['loaded_key']] = new_state
            processed_partitions.append(
                {
                    'source_name': partition_record['source_name'],
                    'partition_value': partition_record['partition_value'],
                    'action': 'loaded',
                    'rows_loaded': rows_loaded,
                }
            )

    # Возвращаем сводку прогона для печати в notebook и для тестов.
    return {
        'sources_root': str(resolved_sources_root),
        'warehouse_root': str(warehouse_path),
        'loaded_partitions': loaded_partitions,
        'skipped_partitions': skipped_partitions,
        'failed_partitions': failed_partitions,
        'processed_partitions': processed_partitions,
        'table_locations': table_locations,
    }


## 5. Финальный запуск на реальных источниках

Финальный прогон использует parquet-источники из `data/sources` и записывает результат в стандартный каталог `warehouse_spark_sql`.


In [ ]:
# Этот блок запускает notebook как законченный ETL-сценарий на реальных parquet-источниках.
if os.environ.get('DBSCORING_SKIP_FINAL_RUN', '0') == '1':
    print('Финальный прогон пропущен по флагу DBSCORING_SKIP_FINAL_RUN=1')
else:
    # Поднимаем отдельную Spark-сессию для финального прогона.
    final_spark = create_spark_session(
        app_name='spark-lab-final-run',
        warehouse_dir=Path(os.environ.get('DBSCORING_WAREHOUSE_ROOT', 'warehouse_spark_sql')) / '_metastore',
        shuffle_partitions=2,
    )
    try:
        # Основная ETL-функция возвращает сводку по загруженным, пропущенным и ошибочным партициям.
        final_summary = run_warehouse_update(
            final_spark,
            sources_root=os.environ.get('DBSCORING_SOURCES_ROOT'),
            warehouse_root=os.environ.get('DBSCORING_WAREHOUSE_ROOT'),
        )
        print(json.dumps(final_summary, ensure_ascii=False, indent=2))
    finally:
        final_spark.stop()


## 6. Первые 10 строк таблиц хранилища


In [ ]:
# Этот блок нужен только для визуального контроля результатов после final_run.
if 'final_summary' not in globals():
    print('Предпросмотр таблиц пропущен: сначала выполните final_run без DBSCORING_SKIP_FINAL_RUN=1')
else:
    # Используем отдельную preview-сессию, чтобы независимо читать итоговые parquet-таблицы.
    preview_spark = create_spark_session(
        app_name='spark-lab-preview',
        warehouse_dir=Path(final_summary['warehouse_root']) / '_preview_metastore',
        shuffle_partitions=2,
    )
    try:
        # Последовательно показываем основные бизнес- и технические таблицы хранилища.
        for table_name in ('dim_sources', 'dim_attributes', 'load_log', 'tech_source_partitions', 'client_monthly_attrs_scd1', 'client_daily_attrs_scd2'):
            print(f'\n=== {table_name} ===')
            read_warehouse_table(preview_spark, table_name, final_summary['table_locations']).show(10, truncate=False)
    finally:
        preview_spark.stop()
